# RA2CE Notebook to get graphs and perform  hazard overlay

Do your imports

In [2]:
# === Standard Library ===
from pathlib import Path
import pickle

# === Scientific & Data Libraries ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm


# === Geospatial Libraries ===
import geopandas as gpd
import rasterio
import folium
from shapely.geometry import box, Point, LineString, Polygon, shape
from pyproj import Transformer
import networkx as nx
import geohexgrid as ghg
from shapely.ops import transform


# === RA2CE Project Imports ===
from ra2ce.network.network_config_data.enums.aggregate_wl_enum import AggregateWlEnum
from ra2ce.network.network_config_data.enums.source_enum import SourceEnum
from ra2ce.network.network_config_data.enums.network_type_enum import NetworkTypeEnum
from ra2ce.network.network_config_data.enums.road_type_enum import RoadTypeEnum
from ra2ce.network.network_config_data.network_config_data import (
    HazardSection,
    NetworkConfigData,
    NetworkSection,
    OriginsDestinationsSection
)
from ra2ce.network.exporters.geodataframe_network_exporter import GeoDataFrameNetworkExporter
from ra2ce.network.exporters.multi_graph_network_exporter import MultiGraphNetworkExporter
from ra2ce.network.network_wrappers.osm_network_wrapper.osm_network_wrapper import OsmNetworkWrapper
from ra2ce.ra2ce_handler import Ra2ceHandler

import os
from pathlib import Path


In [5]:
# specify the name of the path to the project folder where you created the RA2CE folder setup

root_dir = Path(r'C:\python\powerpath\data')
#root_dir = Path.cwd().parent.joinpath("data")
assert root_dir.exists()
static_path = root_dir.joinpath("static")
network_path = static_path.joinpath("network")
output_path = static_path.joinpath("output_graph")
hazard_path = static_path.joinpath(r"C:\python\Use_case_stedin\hazard_maps_time_series_waterbom\hazard_timeseries\reprojected")


## Find the study area

In [6]:
Extent_path = network_path.joinpath("try_study_area_larger.shp")
Extent = gpd.read_file(Extent_path, driver='ESRI Shapefile')
shapely_polygon = Extent.geometry.iloc[0]

# Data pre-processing

In [8]:
# some preliminary functions

def get_all_files(directory: str) -> list[Path]:
    p = Path(directory)
    return [file for file in p.iterdir() if file.is_file()]

def read_pickle(file_path: str):
    with open(file_path, 'rb') as file:c
    data = pickle.load(file)
    return data

def read_gpkg_to_gdf(file_path: str, layer: str = None) -> gpd.GeoDataFrame:
    # Read the geopackage file into a GeoDataFrame
    gdf = gpd.read_file(file_path, layer=layer)
    return gdf

In [32]:
hazard_files = get_all_files(hazard_path)
hazard_crs = "EPSG:4326" # for the hackathon case => "EPSG:4326" 

for hazard_file in hazard_files:
    print (hazard_file)

hazard_files

C:\python\Use_case_stedin\hazard_maps_time_series_waterbom\hazard_timeseries\reprojected\20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-03_scale_5_wgs84.tif
C:\python\Use_case_stedin\hazard_maps_time_series_waterbom\hazard_timeseries\reprojected\20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-04_scale_5_wgs84.tif
C:\python\Use_case_stedin\hazard_maps_time_series_waterbom\hazard_timeseries\reprojected\20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-05_scale_5_wgs84.tif
C:\python\Use_case_stedin\hazard_maps_time_series_waterbom\hazard_timeseries\reprojected\20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-06_scale_5_wgs84.tif
C:\python\Use_case_stedin\hazard_maps_time_series_waterbom\hazard_timeseries\reprojected\20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-07_scale_5_wgs84.tif
C:\python\Use_case_stedin\hazard_maps_time_series_waterbom\hazard

[WindowsPath('C:/python/Use_case_stedin/hazard_maps_time_series_waterbom/hazard_timeseries/reprojected/20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-03_scale_5_wgs84.tif'),
 WindowsPath('C:/python/Use_case_stedin/hazard_maps_time_series_waterbom/hazard_timeseries/reprojected/20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-04_scale_5_wgs84.tif'),
 WindowsPath('C:/python/Use_case_stedin/hazard_maps_time_series_waterbom/hazard_timeseries/reprojected/20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-05_scale_5_wgs84.tif'),
 WindowsPath('C:/python/Use_case_stedin/hazard_maps_time_series_waterbom/hazard_timeseries/reprojected/20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-06_scale_5_wgs84.tif'),
 WindowsPath('C:/python/Use_case_stedin/hazard_maps_time_series_waterbom/hazard_timeseries/reprojected/20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-07_s

# RA2CE

### Let us first initalize and perform the ra2ce run so we have all the data that we need

#### Cutting RoadTypeEnum.MOTORWAY,RoadTypeEnum.MOTORWAY_LINK to make analysis more realistic

In [35]:
_network_section = NetworkSection(
    network_type=NetworkTypeEnum.DRIVE,
    source=SourceEnum.OSM_DOWNLOAD,
    polygon=Extent_path, #it needs a path without the list!
    save_gpkg=True,
    road_types=[RoadTypeEnum.MOTORWAY, RoadTypeEnum.MOTORWAY_LINK, RoadTypeEnum.PRIMARY, RoadTypeEnum.PRIMARY_LINK,RoadTypeEnum.TRUNK, RoadTypeEnum.SECONDARY,RoadTypeEnum.SECONDARY_LINK, RoadTypeEnum.TERTIARY, RoadTypeEnum.RESIDENTIAL] 
    #attributes_to_exclude_in_simplification=['bridge', 'tunnel'],
)

# Make the NetworkConfigData
_hazard_section = HazardSection(
    hazard_map=hazard_files,
    hazard_id=None,
    hazard_field_name="waterdepth",
    aggregate_wl=AggregateWlEnum.MAX,
    hazard_crs=hazard_crs,
    overlay_segmented_network = False
)

_network_config_data = NetworkConfigData(
    root_path=root_dir,
    static_path=static_path,
    output_path=output_path,
    network=_network_section,
    hazard=_hazard_section)


In [36]:
# Run analysis
_handler = Ra2ceHandler.from_config(_network_config_data, analysis=None)
_handler.configure()


Graph hazard overlay with 20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-03_scale_5_wgs84: 100%|██████████| 8694/8694 [00:47<00:00, 184.88it/s]
Graph fraction with hazard overlay with 20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-03_scale_5_wgs84: 100%|██████████| 8694/8694 [01:16<00:00, 113.44it/s]
Graph hazard overlay with 20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-04_scale_5_wgs84: 100%|██████████| 8694/8694 [00:53<00:00, 163.59it/s]
Graph fraction with hazard overlay with 20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-04_scale_5_wgs84: 100%|██████████| 8694/8694 [01:29<00:00, 97.61it/s] 
Graph hazard overlay with 20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_waterschijf_1900-01-05_scale_5_wgs84: 100%|██████████| 8694/8694 [00:48<00:00, 179.52it/s]
Graph fraction with hazard overlay with 20220610 JF  Limburgbui 200 mm 48h  voormalen 15 cm  GHG_water